# Heterogeneous knowledge graph

Connect the independent branches through shared industrial classes.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/config.py").exists())
sys.path.insert(0, str(ROOT))
from src.data import write_json

pd.set_option("display.max_columns", 20)

import networkx as nx
from src.knowledge_graph import build_sensor_branch, integrate_ontology, integration_checks, to_graph

text_nodes = pd.read_csv(ROOT / "results/kg/text/nodes.csv", keep_default_na=False)
text_edges = pd.read_csv(ROOT / "results/kg/text/edges.csv", keep_default_na=False)
events = pd.read_csv(ROOT / "results/sensor/events.csv", keep_default_na=False)
out = ROOT / "results/kg"
sensor_nodes, sensor_edges = build_sensor_branch(events)
nodes, edges = integrate_ontology(text_nodes, text_edges, sensor_nodes, sensor_edges)
checks = integration_checks(nodes, edges, events)
checks.to_csv(out / "integration_checks.csv", index=False)
assert checks["passed"].all(), checks.loc[~checks["passed"]].to_dict("records")
display(checks)

<local environment>:17: DtypeWarning: Columns (9: span_start, 10: span_end) have mixed types. Specify dtype option on import or set low_memory=False.
  text_edges = pd.read_csv(ROOT / "results/kg/text/edges.csv", keep_default_na=False)


,check,passed
0,sensor events loaded,True
1,text KG loaded,True
2,ontology bridge exists,True
3,no direct cross-dataset edges in either direction,True
4,no identity relations,True
5,every edge has provenance and evidence,True
6,all endpoints exist,True


## Ontology bridge

MetroPT APU is an instance of Compressor. Textual compressor mentions denote the same class, including the air-compressor subclass. The bridge supports conceptual navigation; it supplies no ground-truth sensor-to-maintenance links.

In [2]:
nodes.to_csv(out / "nodes.csv", index=False)
edges.to_csv(out / "edges.csv", index=False)
graph = to_graph(nodes, edges)
nx.write_graphml(graph, out / "industrial_kg.graphml")
write_json(out / "summary.json", {
    "nodes": len(nodes), "edges": len(edges), "sensor_events": len(events),
    "text_work_orders": int(text_nodes["node_type"].eq("work_order").sum()),
    "checks_passed": int(checks["passed"].sum()), "checks_total": len(checks),
    "scope": "ontology-only integration of independent datasets; no record-level asset links"
})
display(nodes["node_type"].value_counts())
display(edges["relation"].value_counts())

node_type
work_order            48613
equipment              9400
sensor_event           2934
asset_mention           168
maintenance_type         74
component                18
sensor                   15
maintenance_action       13
concept                  11
equipment_type            9
failure_mode              8
order_type                8
sensor_family             4
source_dataset            2
sensor_asset              1
Name: count, dtype: int64

relation
HAS_MAINTENANCE_ACTION     167336
INVOLVES_COMPONENT         143530
DERIVED_FROM                48614
HAS_ORDER_TYPE              47892
HAS_MAINTENANCE_TYPE        47873
MENTIONS_ASSET_TOKEN        47662
ABOUT_EQUIPMENT             47562
MENTIONS_EQUIPMENT_TYPE     34115
REPORTS_FAILURE_MODE        18046
INVOLVES_SENSOR             13323
OBSERVED_ON                  2934
PEAK_SENSOR                  2934
HAS_SENSOR                     15
HAS_SENSOR_FAMILY              15
DENOTES                         9
SUBCLASS_OF                     4
INSTANCE_OF                     1
Name: count, dtype: int64